In [12]:
import json
import numpy as np
import pandas as pd

from uniguide.config import get_path, load_config
from uniguide.evaluation.protocol import evaluate_model
from uniguide.models.popularity import PopularityRecommender
from uniguide.models.content_based import ContentBasedRecommender
from uniguide.models.popularity import PopularityRecommender


cfg = load_config()
p = get_path("processed")
train, val, test = (pd.read_parquet(p / f"{n}.parquet") for n in ["train", "val", "test"])
courses = pd.read_parquet(p / "courses.parquet")
cold_courses = set(json.loads((p / "cold_courses.json").read_text()))

# Main comparison: the candidate catalog excludes cold-start courses (they never appear in test)
courses_warm = courses[~courses["course_id"].isin(cold_courses)].reset_index(drop=True)
K = tuple(cfg["evaluation"]["k_values"])
names = courses.set_index("course_id")[["name", "institution"]]


def mean_metric(res, metric, k):
    """Mean of one metric at one K over all evaluated users."""
    return res[(res["metric"] == metric) & (res["k"] == k)]["value"].mean()

In [13]:
cb = ContentBasedRecommender().fit(train, courses_warm)
terms = np.array(cb.vectorizer.get_feature_names_out())

def top_terms(course_id, n=10):
    row = cb.item_vectors[cb.item_index[course_id]].toarray().ravel()
    return terms[row.argsort()[::-1][:n]].tolist()

print(cb.item_vectors.shape)
top_terms("machine-learning")

(380, 20000)


['machine learning',
 'andrew',
 'machine',
 'programming exercises',
 'algorithms',
 'lot sir',
 'advanced concepts',
 'algebra',
 'andrew great',
 'basics machine']

In [14]:
# Most similar courses to a given course (should be thematically close)
cid = "machine-learning"
sim = (cb.item_vectors @ cb.item_vectors[cb.item_index[cid]].T).toarray().ravel()
best = np.argsort(-sim)[1:6]
pd.DataFrame({"course_id": [cb.catalog[i] for i in best], "cosine": sim[best]}).join(names, on="course_id")

,course_id,cosine,name,institution
0,neural-networks-deep-learning,0.211106,Neural Networks and Deep Learning,DeepLearning.AI
1,ml-foundations,0.178090,Machine Learning Foundations: A Case Study App...,University of Washington
2,convolutional-neural-networks,0.165639,Convolutional Neural Networks,DeepLearning.AI
3,linear-algebra-machine-learning,0.161936,Mathematics for Machine Learning: Linear Algebra,Imperial College London
4,pca-machine-learning,0.156814,Mathematics for Machine Learning: PCA,Imperial College London


In [15]:
sample_users = val["user_id"].drop_duplicates().sample(3000, random_state=cfg["seed"])
val_s = val[val["user_id"].isin(sample_users)]

grid = [
    dict(ngram_range=(1, 1), max_df=0.5),
    dict(ngram_range=(1, 2), max_df=0.5),
    dict(ngram_range=(1, 2), max_df=0.8),
    dict(ngram_range=(1, 2), max_df=0.5, rating_offset=3),
]
rows = []
for params in grid:
    model = ContentBasedRecommender(**params).fit(train, courses_warm)
    res = evaluate_model(model, val_s, k_values=K)
    rows.append({**params, "ndcg@10": mean_metric(res, "ndcg", 10),
                 "hit_rate@10": mean_metric(res, "hit_rate", 10)})
tuning_cb = pd.DataFrame(rows)
tuning_cb

,ngram_range,max_df,ndcg@10,hit_rate@10,rating_offset
0,"(1, 1)",0.5,0.100242,0.185567,NaN
1,"(1, 2)",0.5,0.105430,0.192321,NaN
2,"(1, 2)",0.8,0.104520,0.193743,NaN
3,"(1, 2)",0.5,0.104992,0.191966,3.0


In [16]:
best_params = grid[int(tuning_cb["ndcg@10"].idxmax())]
tuning_cb.to_csv(get_path("results") / "tables" / "tuning_content_based.csv", index=False)
best_params

{'ngram_range': (1, 2), 'max_df': 0.5}

In [17]:
train_val = pd.concat([train, val])
cb = ContentBasedRecommender(**best_params).fit(train_val, courses_warm)
res_cb = evaluate_model(cb, test, k_values=K, experiment="main", seed=cfg["seed"])
res_cb.to_csv(get_path("results") / "tables" / "eval_content_based.csv", index=False)
print(res_cb.attrs)
res_cb.groupby(["k", "metric"])["value"].mean().unstack()

{'evaluated_users': 14039, 'excluded_users_without_relevant_items': 717}


metric,hit_rate,ndcg,precision,recall
k,,,,
5,0.149726,0.082292,0.031099,0.115786
10,0.204217,0.096950,0.021611,0.159082


In [18]:
res_pop = pd.read_csv(get_path("results") / "tables" / "eval_popularity.csv")
both = pd.concat([res_pop, res_cb])
both.groupby(["method", "k", "metric"])["value"].mean().unstack("metric")

metric            hit_rate      ndcg  precision    recall
method        k                                          
content_based 5   0.149726  0.082292   0.031099  0.115786
              10  0.204217  0.096950   0.021611  0.159082
popularity    5   0.182135  0.081818   0.041684  0.117964
              10  0.279008  0.109250   0.032972  0.196642

In [19]:
user = test["user_id"].iloc[0]
liked = train_val[(train_val["user_id"] == user) & (train_val["rating"] >= 4)]
print("Liked courses:")
display(liked.join(names, on="course_id")[["course_id", "name", "rating"]])
print("Recommended:")
display(cb.recommend(user, k=10).join(names, on="course_id"))

Liked courses:


,course_id,name,rating
0,neural-networks-deep-learning,Neural Networks and Deep Learning,5
1,deep-neural-network,Improving Deep Neural Networks: Hyperparameter...,5
0,machine-learning-projects,Structuring Machine Learning Projects,5


Recommended:


,course_id,score,name,institution
0,introduction-tensorflow,0.276894,Introduction to TensorFlow for Artificial Inte...,DeepLearning.AI
1,convolutional-neural-networks,0.255999,Convolutional Neural Networks,DeepLearning.AI
2,machine-learning,0.245826,Machine Learning,Stanford University
3,convolutional-neural-networks-tensorflow,0.227086,Convolutional Neural Networks in TensorFlow,DeepLearning.AI
4,natural-language-processing-tensorflow,0.219251,Natural Language Processing in TensorFlow,DeepLearning.AI
5,ml-foundations,0.204818,Machine Learning Foundations: A Case Study App...,University of Washington
6,machine-learning-duke,0.192639,Introduction to Machine Learning,Duke University
7,tensorflow-sequences-time-series-and-prediction,0.190954,"Sequences, Time Series and Prediction",DeepLearning.AI
8,python-machine-learning,0.162474,Applied Machine Learning in Python,University of Michigan
9,multivariate-calculus-machine-learning,0.159651,Mathematics for Machine Learning: Multivariate...,Imperial College London


In [20]:
ids = courses_warm["course_id"].tolist()
scores = cb.score_from_ratings({"machine-learning": 5}, ids)
(pd.DataFrame({"course_id": ids, "score": scores})
   .sort_values("score", ascending=False).head(10).join(names, on="course_id"))

,course_id,score,name,institution
0,machine-learning,1.000000,Machine Learning,Stanford University
8,neural-networks-deep-learning,0.211106,Neural Networks and Deep Learning,DeepLearning.AI
361,ml-foundations,0.178090,Machine Learning Foundations: A Case Study App...,University of Washington
70,convolutional-neural-networks,0.165639,Convolutional Neural Networks,DeepLearning.AI
44,linear-algebra-machine-learning,0.161936,Mathematics for Machine Learning: Linear Algebra,Imperial College London
375,pca-machine-learning,0.156814,Mathematics for Machine Learning: PCA,Imperial College London
273,computational-neuroscience,0.155980,Computational Neuroscience,University of Washington
114,machine-learning-with-python,0.152895,Machine Learning with Python,IBM
207,multivariate-calculus-machine-learning,0.151050,Mathematics for Machine Learning: Multivariate...,Imperial College London
195,algorithmic-toolbox,0.150849,Algorithmic Toolbox,University of California San Diego
